In [15]:
!pip uninstall -y transformers
!pip install -U transformers accelerate bitsandbytes sentencepiece

Found existing installation: transformers 5.11.0
Uninstalling transformers-5.11.0:
  Successfully uninstalled transformers-5.11.0
  Using cached transformers-5.11.0-py3-none-any.whl.metadata (33 kB)
Using cached transformers-5.11.0-py3-none-any.whl (11.1 MB)


In [16]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import pandas as pd
import re

In [17]:
def extract_final_num(text):
    match = re.search(r"Final\s+Answer:\s*(-?\d+\.?\d*)", text, re.IGNORECASE)
    if match:
        return match.group(1).strip('.')
    nums = re.findall(r'-?\d+\.?\d*', text)
    if nums:
        return nums[-1].strip('.')
    return None

In [18]:
test_df = pd.read_csv("unified_svamp_test.csv").head(50)

In [19]:
import os
os.environ['LD_LIBRARY_PATH'] += ':/usr/local/cuda-13.0/lib64'

In [20]:
from google.colab import userdata
from huggingface_hub import login

In [21]:
hf_token = userdata.get('HF_TOKEN')
login(hf_token)
print("Authenticated successfully!")

Authenticated successfully!


In [22]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
model_id = "microsoft/Phi-3-mini-4k-instruct"
bnb_config = BitsAndBytesConfig(
    load_in_4bit = True,
    bnb_4bit_quant_type = "nf4",
    bnb_4bit_compute_dtype=torch.float16
)

In [23]:
tokenizer = AutoTokenizer.from_pretrained(model_id, add_bos_token=False)
stop_tokens = ["<|end|>", "<|endoftext|>"]
eos_token_ids = [tokenizer.convert_tokens_to_ids(t) for t in stop_tokens if tokenizer.convert_tokens_to_ids(t) is not None]

In [24]:
print("Downloading the model")
model = AutoModelForCausalLM.from_pretrained(model_id, quantization_config=bnb_config, device_map="auto")
print("Sucess! The engine is running")

Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
results = []
correct_count = 0
total_count = len(test_df)

In [ ]:
print("\nStarting Baseline Evaluation-->")
for index, row in test_df.iterrows():
    question = row['question']
    ground_truth = str(row['answer']).strip()
    messages = [
    {"role": "user", "content": f"You are an expert math tutor. Question: {question}\nAnswer the question step-by-step. You must strictly end your response with 'Final Answer: <number>'."}
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    outputs = model.generate(
        **inputs,
        max_new_tokens=300,
        temperature=0.1,
        do_sample=False,
        eos_token_id=eos_token_ids,
    )
    response = tokenizer.decode(outputs[0][inputs['input_ids'].shape[-1]:], skip_special_tokens=True)
    extracted_answer = extract_final_num(response)

    is_correct = False
    if extracted_answer is not None:
        try:
            if float(extracted_answer) == float(ground_truth):
                is_correct = True
                correct_count += 1
        except ValueError:
            pass

    print(f"\n--- Problem {index + 1} ---")
    print(f"Question: {question}")
    print(f"Model Output:\n{response.strip()}")
    print(f"Extracted Answer: {extracted_answer}")
    print(f"Ground Truth Answer: {ground_truth}")
    print(f"Correct: {is_correct}")
    print("-" * 30)

In [ ]:
accuracy = (correct_count / total_count) * 100
print(f"FINAL BASELINE ACCURACY: {accuracy:.2f}% ({correct_count}/{total_count})")